In [1]:
import pandas as pd
from helpers import get_factor, get_price
pd.options.mode.chained_assignment = None

In [2]:
CDF = pd.read_csv("../production-v2/CDF.csv")
raw_sse = pd.read_csv("../basic/inspire_prtr_mapper.csv")
see = raw_sse.rename(columns={"InspireID_Betrieb": "plantid"})
seem = see[['plantid', 'sseid']]
#bpm['plantid'] = bpm['plantid'].apply(lambda x: str(x).replace('/', '_'))
seem['plantid'] = seem['plantid'].apply(lambda x: str(x).replace('/', '_'))

In [3]:
#CDF.sort_values(by=["produced_at", "variable"])

In [4]:
smard = pd.read_csv("Gro_handelspreise_202401010000_202501010000_Viertelstunde.csv", sep=";", na_values="-", decimal=",", thousands=".")
smardlog = smard[["Datum von", "Deutschland/Luxemburg [€/MWh] Originalauflösungen"]]
smardlog.rename(columns={"Datum von": "timestamp", "Deutschland/Luxemburg [€/MWh] Originalauflösungen": "price"}, inplace=True)
smardlog["timestamp"] = pd.to_datetime(smardlog["timestamp"], format="mixed")

In [5]:
smardlog.dtypes

timestamp    datetime64[us]
price               float64
dtype: object

In [6]:
#smardlog.to_excel("GHP_2023.xlsx")

In [7]:
smardlog.describe()

,timestamp,price
count,35136,35136.000000
mean,2024-07-02 00:26:55.573770496,78.512033
min,2024-01-01 00:00:00,-135.450000
25%,2024-04-01 12:56:15,55.560000
50%,2024-07-02 00:52:30,79.585000
75%,2024-10-01 12:48:45,101.340000
max,2024-12-31 23:45:00,936.280000
std,NaN,52.724158


In [8]:
smardlog.sort_values('price')

,timestamp,price
12721,2024-12-05 13:15:00,-135.45
12722,2024-12-05 13:30:00,-135.45
12723,2024-12-05 13:45:00,-135.45
12720,2024-12-05 13:00:00,-135.45
12725,2024-12-05 14:15:00,-132.85
...,...,...
29831,2024-06-11 17:45:00,820.11
33287,2024-12-12 17:45:00,936.28
33286,2024-12-12 17:30:00,936.28
33285,2024-12-12 17:15:00,936.28


In [9]:
co2s = pd.read_csv("../pollution/pollutants.csv")
nat_mp = pd.read_csv("nat_mapper_2025.csv")
plantlist = pd.read_csv("../basic/plants_2.csv")
nat_mp.fillna(0, inplace=True)

In [10]:
#co2s['year'] = co2s['year'] + 1

In [11]:
#co2s

In [12]:
CDF2 = CDF.loc[CDF.produced_at > "2023-12-31 23:50"].loc[CDF.produced_at < "2025-01-01 00:00"]

In [13]:
CDF2['produced_at'] = pd.to_datetime(CDF2['produced_at'])

In [14]:
CDF2

,produced_at,variable,value
78879,2024-01-01 00:00:00,SEE913896693631,0.0
78880,2024-01-01 01:00:00,SEE913896693631,0.0
78881,2024-01-01 02:00:00,SEE913896693631,0.0
78882,2024-01-01 03:00:00,SEE913896693631,0.0
78883,2024-01-01 04:00:00,SEE913896693631,0.0
...,...,...,...
43493831,2024-12-31 22:45:00,SEE960652233358,0.0
43493832,2024-12-31 23:00:00,SEE960652233358,0.0
43493833,2024-12-31 23:15:00,SEE960652233358,0.0
43493834,2024-12-31 23:30:00,SEE960652233358,0.0


In [15]:
CDF2a = CDF2.copy()#loc[~(CDF2.variable.str.contains("Unnamed"))]

In [16]:
#CDF2b = CDF2a.groupby('variable').resample('1h', on='produced_at').mean()

In [17]:
CDF3 = CDF2a

In [18]:
#CDF3 = CDF2.dropna()

In [19]:
len(CDF2) - len(CDF3)

0

In [20]:
len(CDF2.groupby('produced_at').sum())

35132

In [21]:
dataset = CDF3.merge(seem, left_on="variable", right_on="sseid", how="inner")

In [22]:
dataset2 = dataset.drop_duplicates(subset=['produced_at', 'variable'])

In [23]:
len(dataset)

3750341

In [72]:
dataset.sort_values('value', ascending=False)

,produced_at,variable,value,plantid,sseid
2251051,2024-04-18 12:00:00,SEE925599434282,1203.0,NW100-0248923,SEE925599434282
2256884,2024-12-17 13:00:00,SEE925599434282,1080.0,NW100-0248923,SEE925599434282
2256767,2024-12-12 16:00:00,SEE925599434282,1075.0,NW100-0248923,SEE925599434282
2256802,2024-12-14 03:00:00,SEE925599434282,1074.0,NW100-0248923,SEE925599434282
2256769,2024-12-12 18:00:00,SEE925599434282,1073.0,NW100-0248923,SEE925599434282
...,...,...,...,...,...
1426599,2024-06-05 10:00:00,SEE952372080091,0.0,BYS00041,SEE952372080091
1426600,2024-06-05 11:00:00,SEE952372080091,0.0,BYS00041,SEE952372080091
1426601,2024-06-05 12:00:00,SEE952372080091,0.0,BYS00041,SEE952372080091
1426602,2024-06-05 13:00:00,SEE952372080091,0.0,BYS00041,SEE952372080091


In [25]:
magic = dataset2.groupby(["produced_at", "plantid"]).sum()

In [26]:
#magic

In [27]:
magic2 = magic[["value"]]

In [28]:
#magic2.sort_values(["produced_at", "value"])

In [29]:
production = magic2.reset_index()
production["produced_at"] = pd.to_datetime(production["produced_at"], format="mixed")

In [30]:
merged = production.merge(smardlog, left_on="produced_at", right_on="timestamp")
merged["revenue"] = merged["value"] * merged["price"]

In [31]:
merged_mytmp = merged[['produced_at', 'plantid', 'value', 'price', 'revenue']]

In [32]:
merged_mytmp.loc[merged_mytmp.plantid == "BB45025564"].sort_values('revenue').resample('1YE', on='produced_at').sum()

,plantid,value,price,revenue
produced_at,,,,
2024-12-31,BB45025564BB45025564BB45025564BB45025564BB4502...,9559700.0,2758598.8,8.005455e+08


In [33]:
#merged1a = merged.groupby('plantid').resample('1h', on='produced_at').mean().reset_index()

In [34]:
#merged1a.sort_values('revenue')

In [35]:
#merged

In [36]:
#production.sort_values(by=["plantid", "produced_at"])

In [37]:
#merged1a

In [38]:

#merged_tmp = merged.drop(columns=["timestamp"])
#merged1a = merged_tmp.set_index("produced_at", drop=True)


In [39]:
#merged1a = merged_tmp.set_index(['plantid']).sort_values(['plantid', 'produced_at'])

In [40]:
#merged2 = merged1a.resample("1h", on="produced_at").agg({'value':'sum', 'price':'sum', 'revenue': 'sum' })

In [41]:
#merged1a

In [42]:
#tmp1 = merged2.copy()
tmp1 = merged[["plantid", "value", "price", "revenue"]].groupby("plantid").sum()

In [43]:
tmp1.reset_index(inplace=True)

In [44]:
revenue = tmp1[["plantid", "revenue"]]

In [45]:
tmp1

,plantid,value,price,revenue
0,06-02-B10117A007,637346.0,689649.7,5.288343e+07
1,BB23020490,1376356.0,2758598.8,1.086602e+08
2,BB45025564,9559700.0,2758598.8,8.005455e+08
3,BB45025611,7007511.0,689649.7,6.055226e+08
4,BE166928,1171476.0,2758598.8,9.504822e+07
5,BE169709,362015.0,689649.7,3.070577e+07
6,BE172654,1297508.0,2758598.8,1.158482e+08
7,BE172656,962853.0,689649.7,8.590997e+07
8,BWpf-450-1020129-00000000,593836.0,2758598.8,6.110538e+07
9,BWpf-450-1195689-00000000,205400.0,2758598.8,1.626545e+07


In [46]:
#dataset.sort_values(["plantid", "produced_at"])

In [47]:
plantlist2 = plantlist[["plantid", "energysource"]]
plantlist3 = plantlist2.merge(nat_mp, on="plantid")

In [48]:
tmp0 = pd.merge(revenue, plantlist3, on="plantid")
tmp0["factor"] = tmp0["energysource"].apply(get_factor)
tmp0["fuel_price"] = tmp0["energysource"].apply(get_price)

In [49]:
#prod2 = prod.loc[prod.year == 2023].loc[prod.yearpower > 1000000]
co2s2 = co2s.loc[co2s.year == 2024].loc[co2s.pollutant == "CO2"]

In [50]:
co2s2.drop_duplicates(subset=["year", "plantid", "pollutant"], inplace=True)

In [51]:
co2s2

,year,plantid,pollutant,releases_to,amount,potency,unit_2,amount_2,pollutant2
55,2024,BB16018798,CO2,Air,2.330000e+08,9,Mio. t,0.233,CO2 [Mio. t]
88,2024,BB23020389,CO2,Air,3.250000e+08,9,Mio. t,0.325,CO2 [Mio. t]
171,2024,BB23020490,CO2,Air,3.277000e+09,9,Mio. t,3.277,CO2 [Mio. t]
387,2024,BB23022811,CO2,Air,1.210000e+08,9,Mio. t,0.121,CO2 [Mio. t]
430,2024,BB45025564,CO2,Air,1.225700e+10,9,Mio. t,12.257,CO2 [Mio. t]
...,...,...,...,...,...,...,...,...,...
15994,2024,ST18046,CO2,Air,2.210000e+08,9,Mio. t,0.221,CO2 [Mio. t]
16127,2024,TH30013152,CO2,Air,2.540000e+08,9,Mio. t,0.254,CO2 [Mio. t]
16160,2024,TH62013494,CO2,Air,1.270000e+08,9,Mio. t,0.127,CO2 [Mio. t]
16238,2024,TH72012874,CO2,Air,1.850000e+08,9,Mio. t,0.185,CO2 [Mio. t]


In [52]:
co2s3 = co2s2[["plantid", "amount_2"]]

In [53]:
tmp1 = pd.merge(tmp0, co2s3, on="plantid")

In [54]:
tmp2 = pd.merge(tmp1, production, on="plantid")

In [55]:
tmp2 = tmp1

In [56]:
coal_cost_per_t = 103.5# or 120
co2_cost = 69 # see below
#co2_cost = 65 # alternative https://icapcarbonaction.com/system/files/ets_pdfs/icap-etsmap-factsheet-43.pdf
#electricity_price = 78.50

#https://www.unternehmensregister.de/de/publication?payload=LXxXcfbIejd_v2Be0PSDkD7vo76lhpnrWMKzXGRjUoEPqWlxLhTxwRcC1GfbHMjobv880NLL8BOogW39iTMtoYMTf3C7JceYGttc9xoAHNkSjhhA1vTajd5nW7vWDfQzL-X9mmnDwZsSkqYbE4IM92VMRa8KT7k6axrG8WFJALfV90eno0kl1qqDGUmTftPZBku3uRcZu-UGVw

In [57]:
tmp2

,plantid,revenue,energysource,plantname,free_co2s,factor,fuel_price,amount_2
0,BB23020490,1.086602e+08,Mineralölprodukte,1MKA,0.0,2.30,75,3.277000
1,BB45025564,8.005455e+08,Braunkohle,Kraftwerk Jänschwalde Block A,11960.0,3.25,18,12.257000
2,BB45025611,6.055226e+08,Braunkohle,Kraftwerk Schwarze Pumpe Block A,207831.0,3.25,18,8.081269
3,BE166928,9.504822e+07,Steinkohle,HKW Reuter West Dampfturbine D,63958.0,2.68,120,1.446000
4,BE169709,3.070577e+07,Erdgas,HKW Klingenberg Dampfturbine 1,82983.0,1.50,40,0.426000
5,BE172654,1.158482e+08,Erdgas,HKW Mitte Dampfturbine,79075.0,1.50,40,0.672000
6,BE172656,8.590997e+07,Erdgas,GuD Marzahn Dampfturbine,22820.0,1.50,40,0.522000
7,BWpf-450-1020129-00000000,6.110538e+07,Steinkohle,Heizkraftwerk Heilbronn FHT 1,0.0,2.68,120,0.773000
8,BWpf-450-1195689-00000000,1.626545e+07,Steinkohle,Heizkraftwerk Stuttgart-Münster DT 12,0.0,2.68,120,0.425000
9,BWpf-450-1741292-00000000,4.878585e+07,Steinkohle,Heizkraftwerk Altbach/Deizisau HKW 1,17065.0,2.68,120,0.766000


In [58]:
tmp2["co2_cost"] = (tmp2["amount_2"] * 10**6 - tmp2["free_co2s"]) * co2_cost / 10**6
tmp2["coal_cost"] = (tmp2["amount_2"] * 10**6 * 1/tmp2["factor"] * tmp2["fuel_price"]) / 10**6

In [59]:
tmp2["profit"] = (tmp2["revenue"] / 10**6) - (tmp2["co2_cost"] + tmp2["coal_cost"])

In [70]:
analysis = tmp2[(tmp2.plantname == "Neurath F") | (tmp2.plantname == "Weisweiler F") | (tmp2.plantname == "Niederaußem G")]

In [71]:
analysis

,plantid,revenue,energysource,plantname,free_co2s,factor,fuel_price,amount_2,co2_cost,coal_cost,profit
25,NW100-0248923,1.093000e+09,Braunkohle,Neurath F,3001.0,3.25,18,13.379,922.943931,74.099077,95.956922
27,NW300-0326774,9.262812e+08,Braunkohle,Niederaußem G,26041.0,3.25,18,11.821,813.852171,65.470154,46.958842
29,NW300-0877384,7.522053e+08,Braunkohle,Weisweiler F,12223.0,3.25,18,10.473,721.793613,58.004308,-27.592588


In [65]:
tmp2.sort_values('coal_cost', ascending=False)

,plantid,revenue,energysource,plantname,free_co2s,factor,fuel_price,amount_2,co2_cost,coal_cost,profit
11,BWpf-450-2948214-00000000,1.721634e+08,Steinkohle,GKM Block 6,92027.0,2.68,120,3.145000,210.655137,140.820896,-179.312603
41,RP5000671,1.576855e+08,Erdgas,GuD Mitte DT10,0.0,1.50,40,5.120000,353.280000,136.533333,-332.127784
0,BB23020490,1.086602e+08,Mineralölprodukte,1MKA,0.0,2.30,75,3.277000,226.113000,106.858696,-224.311473
40,NW900-9141660,2.492269e+08,Steinkohle,Trianel Kohlekraftwerk Lünen,1662.0,2.68,120,2.156000,148.649322,96.537313,4.040292
10,BWpf-450-2797933-00000000,1.949110e+08,Steinkohle,Rheinhafen- Dampfkraftwerk RDK 4S DT,0.0,2.68,120,1.973000,136.137000,88.343284,-29.569291
45,SN70015796,1.069532e+09,Braunkohle,Boxberg Block N,6085.0,3.25,18,13.836000,954.264135,76.630154,38.638132
34,NW500-0915123,1.615442e+08,Steinkohle,Datteln 4,631.0,2.68,120,1.710000,117.946461,76.567164,-32.969410
25,NW100-0248923,1.093000e+09,Braunkohle,Neurath F,3001.0,3.25,18,13.379000,922.943931,74.099077,95.956922
1,BB45025564,8.005455e+08,Braunkohle,Kraftwerk Jänschwalde Block A,11960.0,3.25,18,12.257000,844.907760,67.884923,-112.247154
27,NW300-0326774,9.262812e+08,Braunkohle,Niederaußem G,26041.0,3.25,18,11.821000,813.852171,65.470154,46.958842


In [61]:
#tmp2.sort_values(by="free_co2s", ascending=False)

In [62]:
profit = tmp2[["plantid", "plantname", "revenue", "profit"]]
profit["revenue"] = profit["revenue"].apply(lambda x: x / 10**6)

In [63]:
profit.sort_values("profit", ascending=False)

,plantid,plantname,revenue,profit
15,BYS00048,Irsching 5 DT,359.523220,236.113220
22,NI10257673950,Emsland B DT,282.762309,172.884594
23,NW100-0167182,SWD KWF GTKW,245.622271,151.581937
46,SN80011277,Kraftwerk Lippendorf Block S,532.706238,108.524848
25,NW100-0248923,Neurath F,1092.999930,95.956922
39,NW900-9140178,Trianel Gaskraftwerk Hamm Block 10,137.017067,91.766734
28,NW300-0370387,DT Niehl 2 RheinEnergie,194.223969,84.015969
32,NW300-9046030,Knapsack I - Dampfturbine - DT 10,124.180906,78.260906
31,NW300-9002708,Dormagen DT,131.031108,66.647441
5,BE172654,HKW Mitte Dampfturbine,115.848185,57.016360


In [64]:
profit.sort_values("profit", ascending=False)

,plantid,plantname,revenue,profit
15,BYS00048,Irsching 5 DT,359.523220,241.273220
22,NI10257673950,Emsland B DT,282.762309,177.430162
23,NW100-0167182,SWD KWF GTKW,245.622271,155.513937
25,NW100-0248923,Neurath F,1092.999930,149.460918
46,SN80011277,Kraftwerk Lippendorf Block S,532.706238,131.275020
27,NW300-0326774,Niederaußem G,926.281167,94.138678
45,SN70015796,Boxberg Block N,1069.532421,93.957792
39,NW900-9140178,Trianel Gaskraftwerk Hamm Block 10,137.017067,93.658734
28,NW300-0370387,DT Niehl 2 RheinEnergie,194.223969,88.623969
32,NW300-9046030,Knapsack I - Dampfturbine - DT 10,124.180906,80.180906


In [66]:
#profit_combined = pd.concat([profit, profit2])

In [67]:
#profit_final = profit_combined.drop_duplicates(subset="plantid", keep="first").sort_values('profit')

In [68]:
profit.to_csv("profit.csv", index=False)

In [ ]:
pd.concat([profit2, profit, profit]).drop_duplicates(keep=False)

In [ ]:
#profit_final.reset_index()

In [ ]:
pd.concat([profit, profit2, profit2]).drop_duplicates(keep=False).sort_values('profit', ascending=False)

In [ ]:
#profit2 = profit

In [ ]:
profit2.sort_values("profit", ascending=False)